## Dataset exploration
The Massachusetts Buildings Dataset used in this project contains aerial RGB images and corresponding building segmentation masks. As our dataset already has train, test and val, we simply need to ensure that the data is correct and suitable for further preprocessing.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_DIR = RAW_DIR / "train"
TRAIN_LABELS_DIR = RAW_DIR / "train_labels"

VAL_DIR = RAW_DIR / "val"
VAL_LABELS_DIR = RAW_DIR / "val_labels"

TEST_DIR = RAW_DIR / "test"
TEST_LABELS_DIR = RAW_DIR / "test_labels"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)

Project root: /home/kriss/VSProjects/CV_project/CV-P22-building-segmentation
Raw data directory: /home/kriss/VSProjects/CV_project/CV-P22-building-segmentation/data/raw


In [4]:
data_dirs = {
    "train": TRAIN_DIR,
    "train_labels": TRAIN_LABELS_DIR,
    "val": VAL_DIR,
    "val_labels": VAL_LABELS_DIR,
    "test": TEST_DIR,
    "test_labels": TEST_LABELS_DIR,
}

for name, directory in data_dirs.items():
    if directory.exists():
        num_files = len(list(directory.glob("*.png")))
        print(f"{name:15} | {num_files:5} PNG files")
    else:
        print(f"{name:15} | directory not found")

train           |   137 PNG files
train_labels    |   137 PNG files
val             |     4 PNG files
val_labels      |     4 PNG files
test            |    10 PNG files
test_labels     |    10 PNG files


In [5]:
def check_matching_files(images_dir, labels_dir, split_name):
    image_names = {path.name for path in images_dir.glob("*.png")}
    label_names = {path.name for path in labels_dir.glob("*.png")}

    missing_labels = image_names - label_names
    missing_images = label_names - image_names

    print(f"{split_name}:")
    print(f"  images without masks: {len(missing_labels)}")
    print(f"  masks without images: {len(missing_images)}")

    if not missing_labels and not missing_images:
        print("  all image-mask pairs match")
    print()


check_matching_files(TRAIN_DIR, TRAIN_LABELS_DIR, "train")
check_matching_files(VAL_DIR, VAL_LABELS_DIR, "val")
check_matching_files(TEST_DIR, TEST_LABELS_DIR, "test")

train:
  images without masks: 0
  masks without images: 0
  all image-mask pairs match

val:
  images without masks: 0
  masks without images: 0
  all image-mask pairs match

test:
  images without masks: 0
  masks without images: 0
  all image-mask pairs match



In [7]:
from PIL import Image
import numpy as np

EXPECTED_SIZE = (1500, 1500)
EXPECTED_MODE = "RGB"

splits = {
    "train": (TRAIN_DIR, TRAIN_LABELS_DIR),
    "val": (VAL_DIR, VAL_LABELS_DIR),
    "test": (TEST_DIR, TEST_LABELS_DIR),
}

issues = []

for split_name, (images_dir, masks_dir) in splits.items():
    for image_path in images_dir.glob("*.png"):
        mask_path = masks_dir / image_path.name

        with Image.open(image_path) as image:
            if image.size != EXPECTED_SIZE:
                issues.append(
                    f"{split_name} image {image_path.name}: "
                    f"unexpected size {image.size}"
                )

            if image.mode != EXPECTED_MODE:
                issues.append(
                    f"{split_name} image {image_path.name}: "
                    f"unexpected mode {image.mode}"
                )

        with Image.open(mask_path) as mask:
            if mask.size != EXPECTED_SIZE:
                issues.append(
                    f"{split_name} mask {mask_path.name}: "
                    f"unexpected size {mask.size}"
                )

            if mask.mode != EXPECTED_MODE:
                issues.append(
                    f"{split_name} mask {mask_path.name}: "
                    f"unexpected mode {mask.mode}"
                )

            mask_values = np.unique(np.array(mask))

            if not np.all(np.isin(mask_values, [0, 255])):
                issues.append(
                    f"{split_name} mask {mask_path.name}: "
                    f"unexpected values {mask_values}"
                )


if issues:
    print(f"Found {len(issues)} potential issues:\n")
    for issue in issues:
        print("-", issue)
else:
    print("OK: all images and masks have the expected size, format, and mask values.")

OK: all images and masks have the expected size, format, and mask values.


In [8]:
import pandas as pd

label_classes = pd.read_csv(RAW_DIR / "label_class_dict.csv")
metadata = pd.read_csv(RAW_DIR / "metadata.csv")

print("label_class_dict.csv")
display(label_classes)

print("\nmetadata.csv")
display(metadata.head())

label_class_dict.csv


,name,r,g,b
0,background,0,0,0
1,building,255,255,255



metadata.csv


,image_id,split,image_souce_url,label_source_url,tiff_image_path,tif_label_path,png_image_path,png_label_path
0,22678915_15,train,http://www.cs.toronto.edu/~vmnih/data/mass_bui...,http://www.cs.toronto.edu/~vmnih/data/mass_bui...,tiff/train/22678915_15.tiff,tiff/train_labels/22678915_15.tif,png/train/22678915_15.png,png/train_labels/22678915_15.png
1,22678930_15,train,http://www.cs.toronto.edu/~vmnih/data/mass_bui...,http://www.cs.toronto.edu/~vmnih/data/mass_bui...,tiff/train/22678930_15.tiff,tiff/train_labels/22678930_15.tif,png/train/22678930_15.png,png/train_labels/22678930_15.png
2,22678945_15,train,http://www.cs.toronto.edu/~vmnih/data/mass_bui...,http://www.cs.toronto.edu/~vmnih/data/mass_bui...,tiff/train/22678945_15.tiff,tiff/train_labels/22678945_15.tif,png/train/22678945_15.png,png/train_labels/22678945_15.png
3,22678960_15,train,http://www.cs.toronto.edu/~vmnih/data/mass_bui...,http://www.cs.toronto.edu/~vmnih/data/mass_bui...,tiff/train/22678960_15.tiff,tiff/train_labels/22678960_15.tif,png/train/22678960_15.png,png/train_labels/22678960_15.png
4,22678975_15,train,http://www.cs.toronto.edu/~vmnih/data/mass_bui...,http://www.cs.toronto.edu/~vmnih/data/mass_bui...,tiff/train/22678975_15.tiff,tiff/train_labels/22678975_15.tif,png/train/22678975_15.png,png/train_labels/22678975_15.png


## Dataset summary

The provided dataset split consists of:

- 137 training images and 137 training masks
- 4 validation images and 4 validation masks
- 10 test images and 10 test masks

All image-mask pairs were successfully matched by filename.

A sanity check over all dataset files confirmed that:

- all images have size 1500 × 1500 pixels;
- all images are stored in RGB format;
- all masks have the same spatial dimensions as their corresponding images;
- all masks are stored in RGB format;
- mask pixel values are binary: 0 for background and 255 for buildings.

No missing image-mask pairs or structural outliers were found.